# Dynamic Pydantic Model Construction

This notebook illustrates how to dynamically construct Pydantic v2 models at runtime using `create_model`, configure fields, constraints, inheritance, validators, and build models directly from schema dictionaries and YAML definitions.

### Table of Contents
1. Import Required Libraries
2. Basic Model Creation with `create_model`
3. Defining Fields with Defaults and Validation Constraints
4. Configuring Inheritance and Model Config Dynamically
5. Attaching Custom Validators Dynamically
6. Constructing Models Dynamically from Runtime Schema Dictionaries and YAML

## 1. Import Required Libraries

Import core utilities from `pydantic` (`create_model`, `Field`, `BaseModel`, `field_validator`, `model_validator`) along with typing constructs and the `prefect_yaml.dynamic_models` package.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Literal

from pydantic import (
    BaseModel,
    ConfigDict,
    Field,
    ValidationError,
    create_model,
    field_validator,
)

from prefect_yaml.dynamic_models import (
    build_model_from_spec,
    load_models_from_yaml,
)

print("Libraries imported successfully!")

## 2. Basic Model Creation with `create_model`

`create_model` allows building a new Pydantic `BaseModel` subclass dynamically. Field arguments are supplied as `(type, default_value)` tuples. Fields with `...` (Ellipsis) are required.

In [ ]:
# Dynamically create a User model
DynamicUser = create_model(
    "DynamicUser",
    name=(str, ...),  # Required string field
    age=(int, 25),  # Optional int with default 25
    email=(str | None, None),  # Nullable string with default None
    is_active=(bool, True),  # Bool with default True
)

print(f"Generated class: {DynamicUser}")
print(f"Model fields: {list(DynamicUser.model_fields.keys())}")

# Instantiate the dynamic model
user1 = DynamicUser(name="Alice", age=30)
print("\nInstance 1:", user1.model_dump())

# Demonstrating defaults
user2 = DynamicUser(name="Bob")
print("Instance 2 (with defaults):", user2.model_dump())

# Validation enforces types
try:
    DynamicUser(name="Charlie", age="not-a-number")
except ValidationError as e:
    print("\nExpected validation error caught:")
    print(e)

## 3. Defining Fields with Defaults and Validation Constraints

In Pydantic v2, passing a `Field(...)` instance as the second element of the tuple gives full control over field metadata:
- **Numerical constraints**: `ge`, `gt`, `le`, `lt`
- **String constraints**: `min_length`, `max_length`, `pattern` (regex)
- **Categorical choices**: `Literal["choice_a", "choice_b"]`
- **Mutable defaults**: `default_factory=list` or `default_factory=dict` to avoid shared mutable state
- **Documentation**: `description`, `title`, and `alias`

In [ ]:
# Dynamically construct a pipeline parameters model with constraints
PipelineParams = create_model(
    "PipelineParams",
    # Numerical bounds: 1 <= batch_size <= 100
    batch_size=(
        int,
        Field(default=10, ge=1, le=100, description="Processing batch size (1-100)"),
    ),
    # Float threshold: 0.0 <= threshold <= 1.0
    threshold=(
        float,
        Field(default=0.8, ge=0.0, le=1.0, description="Confidence threshold"),
    ),
    # Regex pattern: semantic versioning format vX.Y
    version=(
        str,
        Field(..., pattern=r"^v\d+\.\d+$", description="Version string matching ^v\\d+\\.\\d+$"),
    ),
    # Categorical choices via Literal
    execution_mode=(
        Literal["fast", "accurate", "debug"],
        Field(default="fast", description="Execution profile"),
    ),
    # Mutable collections safely isolated via default_factory
    tags=(
        list[str],
        Field(default_factory=list, description="Categorization labels"),
    ),
    # Field alias (e.g., snake_case in Python, kebab-case from input source)
    output_dir=(
        Path,
        Field(default=Path("/tmp/output"), alias="output-dir", description="Output destination"),
    ),
)

# Inspect generated schema
print("Generated JSON Schema:")
print(json.dumps(PipelineParams.model_json_schema(), indent=2))

# Valid instantiation (supports alias and defaults)
params = PipelineParams.model_validate(
    {
        "batch_size": 50,
        "version": "v2.1",
        "execution_mode": "accurate",
        "tags": ["prod", "etl"],
        "output-dir": "/var/data/exports",
    }
)
print("\nValidated instance:", params.model_dump())

## 4. Configuring Inheritance and Model Config Dynamically

You can pass:
- `__base__`: Specifies a parent class (or tuple of base classes) so the dynamic model inherits existing fields, methods, and configurations.
- `__config__`: Specifies Pydantic `ConfigDict` flags like `extra="allow"` or `extra="forbid"`, `arbitrary_types_allowed=True`, `frozen=True`, etc.

In [ ]:
# Define a static base model with shared audit attributes
class BaseEntity(BaseModel):
    id: str = Field(..., description="Unique entity ID")
    created_by: str = Field(default="system", description="User or service author")


# Inherit dynamically from BaseEntity with custom ConfigDict
CustomEntity = create_model(
    "CustomEntity",
    __base__=BaseEntity,
    __config__=ConfigDict(
        extra="forbid",  # Disallow unexpected fields
        str_strip_whitespace=True,  # Automatically trim whitespace on strings
        frozen=True,  # Make model instances immutable
    ),
    payload=(dict[str, Any], Field(default_factory=dict)),
    status=(str, "pending"),
)

# Verify inheritance
print(f"Parent classes: {CustomEntity.__bases__}")
print(f"All fields (including inherited): {list(CustomEntity.model_fields.keys())}")

# Instantiate
entity = CustomEntity(id="ENT-001", payload={"key": "val"}, status="  active  ")
print("\nInstantiated immutable entity:")
print("  ID:", entity.id)
print("  Status (trimmed):", repr(entity.status))
print("  Created By:", entity.created_by)

# Extra field rejected due to extra='forbid'
try:
    CustomEntity(id="ENT-002", unknown_field="invalid")
except ValidationError as e:
    print("\nExpected validation error for unknown field:")
    print(e.errors()[0]["msg"])

## 5. Attaching Custom Validators Dynamically

In Pydantic v2, custom validation logic can be dynamically attached in two ways:
1. **Dynamic Base Classes**: Subclassing a model containing `@field_validator` or `@model_validator` decorated classmethods.
2. **`__validators__` argument**: Passing a dictionary of field validator functions directly into `create_model`. Each function takes `(cls, value)` and returns the validated value.

In [ ]:
# Define validator function
def validate_even_number(cls, v: int) -> int:
    if v % 2 != 0:
        raise ValueError(f"Value must be an even number, got {v}")
    return v


def uppercase_string(cls, v: str) -> str:
    return v.upper()


# Attach validators via __validators__ mapping in create_model
ValidatedModel = create_model(
    "ValidatedModel",
    number=(int, ...),
    code=(str, ...),
    __validators__={
        "check_even": field_validator("number")(validate_even_number),
        "ensure_upper": field_validator("code")(uppercase_string),
    },
)

# Test valid inputs
obj = ValidatedModel(number=42, code="abc")
print("Validated model instance:", obj.model_dump())
assert obj.number == 42
assert obj.code == "ABC"

# Test validation failure
try:
    ValidatedModel(number=13, code="xyz")
except ValidationError as e:
    print("\nCaught custom validator error:")
    print(e.errors()[0]["msg"])

## 6. Constructing Models Dynamically from Runtime Schema Dictionaries and YAML

In real-world workflow engines (like `prefect-yaml`), schemas arrive as JSON dictionaries or YAML files.
The `prefect_yaml.dynamic_models` module automates:
1. Converting YAML string types (`str`, `int`, `float`, `path`, `list[int]`, `dict[str, Any]`, `str | None`) into Python types.
2. Mapping constraints (`minimum`, `maximum`, `regex`, `choices`).
3. Handling schema inheritance (`extends`).
4. Pre-coercing CLI string inputs (e.g., `"1, 2, 3"` → `[1, 2, 3]`).
5. Directly referencing existing Python Pydantic models by dotted path!

In [ ]:
# 6.1 Building from a Python schema dictionary using build_model_from_spec
runtime_schema = {
    "dataset_name": {
        "type": "str",
        "required": True,
        "description": "Target dataset name",
    },
    "sample_rate": {
        "type": "float",
        "default": 0.1,
        "minimum": 0.01,
        "maximum": 1.0,
        "description": "Sampling ratio (0.01 to 1.0)",
    },
    "categories": {
        "type": "list[str]",
        "default": ["sales", "returns"],
        "description": "Categories to process",
    },
    "backup_path": {
        "type": "path",
        "default": "/tmp/backup",
        "description": "Local backup directory",
    },
}

DynamicEtlModel = build_model_from_spec("DynamicEtlModel", runtime_schema)
print("Fields in generated model:")
for name, field in DynamicEtlModel.model_fields.items():
    print(f"  {name}: {field.annotation} (default={field.default!r})")

# Instant validation with string coercion (as from CLI --set)
raw_cli_data = {
    "dataset_name": "customer_churn",
    "sample_rate": "0.25",  # string coerced to float
    "categories": "finance, risk",  # comma-delimited string coerced to list[str]
    "backup_path": "/var/backups",  # string coerced to Path
}

etl_instance = DynamicEtlModel.model_validate(raw_cli_data)
print("\nValidated ETL instance:")
print(" ", etl_instance.model_dump())
print("  backup_path type:", type(etl_instance.backup_path).__name__)

### 6.2 Schema Inheritance (`extends`) and Cross-Model References from YAML

`load_models_from_yaml` parses complete YAML definitions featuring presets, `extends` inheritance, and forward cross-model references.

In [ ]:
YAML_SCHEMA = """
presets:
  base_workflow:
    description: "Common metadata for all workflow steps"
    fields:
      run_id:
        type: str
        required: true
      retries:
        type: int
        default: 3
        minimum: 0
        maximum: 10

entities:
  DatabaseConfig:
    fields:
      host:
        type: str
        default: "localhost"
      port:
        type: int
        default: 5432

  DataPipelineConfig:
    extends: base_workflow
    fields:
      target_db:
        type: DatabaseConfig
      tables:
        type: list[str]
        default: []
      enabled:
        type: bool
        default: true
"""

# Load all models with automatic two-pass resolution and inheritance
models = load_models_from_yaml(YAML_SCHEMA)
print("Compiled models from YAML:", list(models.keys()))

DataPipelineModel = models["DataPipelineConfig"]
DatabaseModel = models["DatabaseConfig"]

# Instantiate with nested cross-model validation and inherited fields
pipeline_run = DataPipelineModel.model_validate(
    {
        "run_id": "RUN-2026-001",
        "retries": 5,  # inherited from base_workflow
        "target_db": {"host": "pg.internal", "port": 5433},  # nested DatabaseConfig model
        "tables": "users, orders, events",  # coerced to list
        "enabled": "yes",  # coerced to bool
    }
)

print("\nValidated Pipeline Run:")
print(json.dumps(pipeline_run.model_dump(), indent=2, default=str))